## Linear Alergba

### Vector

In [2]:
class Vector:
    def __init__(self, components):
        self.components = components
        self.dim = len(components)

    def __add__(self, other):
        return Vector([a + b for a, b in zip(self.components, other.components)])

    def __sub__(self, other):
        return Vector([a - b for a, b in zip(self.components, other.components)])

    def dot(self, other):
        return sum(a * b for a, b in zip(self.components, other.components))

    def magnitude(self):
        return sum(x**2 for x in self.components)**0.5

    def normalize(self):
        mag = self.magnitude()
        return Vector([x / mag for x in self.components])
    
    def cosine_similarity(self, other):
        return self.dot(other) / (self.magnitude() * other.magnitude())

    def __repr__(self) -> str:
        return f"Vector({self.components})"

a = Vector([1, 2, 3])
b = Vector([4, 5, 6])

print(f"a + b = {a + b}")
print(f"a dot b = {a.dot(b)}")
print(f"a magnitude = {a.magnitude()}")
print(f"a normalized = {a.normalize()}")
print(f"cosine similarity = {a.cosine_similarity(b)}")




a + b = Vector([5, 7, 9])
a dot b = 32
a magnitude = 3.7416573867739413
a normalized = Vector([0.2672612419124244, 0.5345224838248488, 0.8017837257372732])
cosine similarity = 0.9746318461970762


### Matrix

In [3]:
class Matrix:
    def __init__(self, rows):
        self.rows = [list(row) for row in rows]
        self.shape = (len(self.rows), len(self.rows[0]))

    def __matmul__(self, other):
        if isinstance(other, Vector):
            return Vector([
                sum(self.rows[i][j] * other.components[j] for j in range(len(other.components)))
                for i in range(len(self.rows))
            ])
        
        rows = []
        for i in range(len(self.shape[0])):
            row = []
            for j in range(len(other.shape[1])):
                row.append(
                    sum(self.rows[i][k] * other.rows[k][j] for k in range(self.shape[1]))
                )
            rows.append(row)
        return Matrix(rows)

    def transpose(self):
        return Matrix([
            [self.rows[j][i] for j in range(self.shape[0])]
            for i in range(self.shape[1])
        ])

    def __repr__(self):
        return f"Matrix({self.rows})"

rotation_90 = Matrix([[0, -1], [1, 0]])
point = Vector([3, 1])
rotated = rotation_90 @ point
print(f"Original point: {point}")
print(f"Rotated point: {rotated}")



Original point: Vector([3, 1])
Rotated point: Vector([-1, 3])


### AI Implements

In [4]:
import random

random.seed(42)
weights = Matrix([
    [random.gauss(0, 0.1) for _ in range(3)] for _ in range(2)
])

input_vector = Vector([1.0, 0.5, -0.3])

output_vector = weights @ input_vector

print(f"Input (3D): {input_vector}")
print(f"Output (2D): {output_vector}")
print("This is what a neural network layer does -- matrix multiplication.")

Input (3D): Vector([1.0, 0.5, -0.3])
Output (2D): Vector([-0.019714737127338927, 0.10873956075097067])
This is what a neural network layer does -- matrix multiplication.


### Some base implements

In [6]:
def is_linearly_independent(vectors):
    n = len(vectors)
    dim = len(vectors[0].components)
    mat = Matrix([v.components[:] for v in vectors])
    rows = [row[:] for row in mat.rows]
    rank = 0
    for col in range(dim):
        pivot = None
        for row in range(rank, len(rows)):
            if abs(rows[row][col]) > 1e-10:
                pivot = row
                break
        if pivot is None:
            continue
        rows[rank], rows[pivot] = rows[pivot], rows[rank]
        scale = rows[rank][col]
        rows[rank] = [x / scale for x in rows[rank]]
        for row in range(len(rows)):
            if row != rank and abs(rows[row][col]) > 1e-10:
                factor = rows[row][col]
                rows[row] = [rows[row][j] - factor * rows[rank][j] for j in range(dim)]
        rank += 1
    return rank == n


def project(a, b):
    scalar = a.dot(b) / b.dot(b)
    return Vector([scalar * x for x in b.components])


def gram_schmidt(vectors):
    orthonormal = []
    for v in vectors:
        w = v
        for u in orthonormal:
            proj = project(w, u)
            w = w - proj
        if w.magnitude() < 1e-10:
            continue
        orthonormal.append(w.normalize())
    return orthonormal


v1 = Vector([1, 0, 0])
v2 = Vector([1, 1, 0])
v3 = Vector([1, 1, 1])
basis = gram_schmidt([v1, v2, v3])
for i, u in enumerate(basis):
    print(f"u{i+1} = {u}")
    print(f"  |u{i+1}| = {u.magnitude():.6f}")

print(f"u1 · u2 = {basis[0].dot(basis[1]):.6f}")
print(f"u1 · u3 = {basis[0].dot(basis[2]):.6f}")
print(f"u2 · u3 = {basis[1].dot(basis[2]):.6f}")

u1 = Vector([1.0, 0.0, 0.0])
  |u1| = 1.000000
u2 = Vector([0.0, 1.0, 0.0])
  |u2| = 1.000000
u3 = Vector([0.0, 0.0, 1.0])
  |u3| = 1.000000
u1 · u2 = 0.000000
u1 · u3 = 0.000000
u2 · u3 = 0.000000


### Use NumPy Instead

* Basic implements

In [7]:
import numpy as np

a = np.array([1, 2, 3], dtype = float)
b = np.array([4, 5, 6], dtype = float)

print(f"a + b = {a + b}")
print(f"a dot b = {np.dot(a, b)}")
print(f"|a| = {np.linalg.norm(a):.4f}")
print(f"cosine = {np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)):.4f}")

W = np.random.randn(2, 3) * 0.1
x = np.array([1.0, 0.5, -0.3])
print(f"Wx = {W @ x}")

a + b = [5. 7. 9.]
a dot b = 32.0
|a| = 3.7417
cosine = 0.9746
Wx = [ 0.16816366 -0.12460846]


* Algebra 

In [9]:
import numpy as np

A = np.array([[1, 2], [2, 4]])
print(f"Rank: {np.linalg.matrix_rank(A)}")

a = np.array([3, 4])
b = np.array([1, 0])

proj = (np.dot(a, b) / np.dot(b, b)) * b
print(f"Projection of {a} onto {b}: {proj}")

Q, R = np.linalg.qr(np.random.randn(3, 3))
print(f"Q is orthogonal: {np.allclose(Q @ Q.T, np.eye(3))}")
print(f"R is upper triangular: {np.allclose(R, np.triu(R))}")


Rank: 1
Projection of [3 4] onto [1 0]: [3. 0.]
Q is orthogonal: True
R is upper triangular: True


### QR Decomposition

把矩阵 $A$ 分解为 $A = QR$：

- **$Q$**：列向量两两正交且单位长度（正交矩阵，$Q^\top Q = I$）
- **$R$**：上三角矩阵（对角线以下全为 0）

直觉：对 $A$ 的列做 **Gram-Schmidt** 正交化得到 $Q$；用各列在正交基上的系数拼成 $R$。

常见用途：解线性方程组、最小二乘、特征值算法（QR 迭代）等。

In [10]:
import numpy as np

# 具体矩阵的 QR 分解
A = np.array([
    [1.0, 1.0, 0.0],
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
])

Q, R = np.linalg.qr(A)

print("A =\n", A)
print("\nQ (orthogonal) =\n", Q)
print("\nR (upper triangular) =\n", R)

# 验证 A ≈ Q @ R
print(f"\nA ≈ QR? {np.allclose(A, Q @ R)}")
print(f"QᵀQ = I? {np.allclose(Q.T @ Q, np.eye(3))}")
print(f"R is upper triangular? {np.allclose(R, np.triu(R))}")

# Q 的列两两正交、范数为 1
print(f"\n|q1| = {np.linalg.norm(Q[:, 0]):.6f}")
print(f"q1 · q2 = {np.dot(Q[:, 0], Q[:, 1]):.6e}")
print(f"q1 · q3 = {np.dot(Q[:, 0], Q[:, 2]):.6e}")
print(f"q2 · q3 = {np.dot(Q[:, 1], Q[:, 2]):.6e}")

A =
 [[1. 1. 0.]
 [1. 0. 1.]
 [0. 1. 1.]]

Q (orthogonal) =
 [[-0.70710678  0.40824829 -0.57735027]
 [-0.70710678 -0.40824829  0.57735027]
 [-0.          0.81649658  0.57735027]]

R (upper triangular) =
 [[-1.41421356 -0.70710678 -0.70710678]
 [ 0.          1.22474487  0.40824829]
 [ 0.          0.          1.15470054]]

A ≈ QR? True
QᵀQ = I? True
R is upper triangular? True

|q1| = 1.000000
q1 · q2 = 1.110223e-16
q1 · q3 = -1.110223e-16
q2 · q3 = -8.326673e-17


#### 与 Gram-Schmidt 的关系

对 $A$ 的列向量做 Gram-Schmidt，得到正交列组成 $Q$；  
$R$ 记录「原列在这些正交方向上的投影系数」：

$$
\mathbf{a}_j = r_{1j}\mathbf{q}_1 + r_{2j}\mathbf{q}_2 + \cdots + r_{jj}\mathbf{q}_j
$$

因此矩阵形式就是 $A = QR$。下面用前面手写的 `gram_schmidt` 对照 NumPy 的 $Q$。

In [11]:
# 用自实现 Gram-Schmidt 得到 Q，再与 np.linalg.qr 对比
cols = [Vector(A[:, j].tolist()) for j in range(A.shape[1])]
Q_gs = gram_schmidt(cols)
Q_from_gs = np.column_stack([u.components for u in Q_gs])

Q_np, R_np = np.linalg.qr(A)

print("Q from Gram-Schmidt:\n", Q_from_gs)
print("\nQ from np.linalg.qr:\n", Q_np)

# 列方向可能差一个符号（±），但应张成同一组正交基
agreement = [
    abs(abs(np.dot(Q_from_gs[:, j], Q_np[:, j])) - 1.0) < 1e-8
    for j in range(3)
]
print(f"\nColumns match up to sign? {all(agreement)}")

# R 可由 R = Qᵀ A 恢复（正交矩阵的好处）
R_recovered = Q_np.T @ A
print("\nR recovered as QᵀA:\n", R_recovered)
print(f"Matches np R? {np.allclose(R_recovered, R_np)}")

Q from Gram-Schmidt:
 [[ 0.70710678  0.40824829 -0.57735027]
 [ 0.70710678 -0.40824829  0.57735027]
 [ 0.          0.81649658  0.57735027]]

Q from np.linalg.qr:
 [[-0.70710678  0.40824829 -0.57735027]
 [-0.70710678 -0.40824829  0.57735027]
 [-0.          0.81649658  0.57735027]]

Columns match up to sign? True

R recovered as QᵀA:
 [[-1.41421356e+00 -7.07106781e-01 -7.07106781e-01]
 [-5.55111512e-17  1.22474487e+00  4.08248290e-01]
 [ 0.00000000e+00 -2.22044605e-16  1.15470054e+00]]
Matches np R? True


#### 小应用：用 QR 解线性方程

若 $A$ 可逆且 $A = QR$，则 $Ax = b$ 变成 $QRx = b$。  
先算 $\mathbf{y} = Q^\top b$（因为 $Q^{-1} = Q^\top$），再对上三角 $R$ 回代解 $Rx = y$。

In [12]:
A = np.array([
    [2.0, 1.0, 1.0],
    [1.0, 3.0, 2.0],
    [1.0, 0.0, 0.0],
])
b = np.array([1.0, 2.0, 3.0])

Q, R = np.linalg.qr(A)
y = Q.T @ b
x = np.linalg.solve(R, y)  # R 上三角，回代求解

print(f"x (via QR) = {x}")
print(f"x (direct) = {np.linalg.solve(A, b)}")
print(f"Ax ≈ b? {np.allclose(A @ x, b)}")

x (via QR) = [  3.   9. -14.]
x (direct) = [  3.   9. -14.]
Ax ≈ b? True


### Use Pytorch Instead

In [ ]:
from networkx.algorithms import similarity
import torch

x = torch.randn(3, requires_grad=True)
y = torch.tensor([1.0, 0.0, 0.0])

similarity = torch.dot(x, y)
similarity.backward()

print(f"x = {x.data}")
print(f"y = {y.data}")
print(f"dot product = {similarity.item():.4f}")
print(f"d(dot)/dx = {x.grad}")